# Synthetic Patient Data Generator

## Purpose

This notebook generates a **unified synthetic patient dataset** for testing the complete patient master data pipeline:

1. **Record Linkage** — Can we correctly identify which records belong to the same person?
2. **Survivorship** — Can we select the "best" record values for each patient?

## Dataset Features

The generated data mirrors real-world messy healthcare data:

- **Multiple temporal records** per entity across different quarters
- **Data entry errors**: typos, character swaps, case variations
- **Legitimate changes over time**: address moves, name updates
- **VSDM verification** data for Tier 1 survivorship selection
- **Varying completeness**: some records missing fields
- **Source system differences**: BEARBEITET, UNBEARBEITET, KVUEPP

## Output Schema

| Field | Description |
|-------|-------------|
| **entity_id** | Ground truth: records with same ID = same person |
| **EGKVERSICHERTENNUMMER** | German health insurance number |
| **VORNAME** | First name (may have typos/variations) |
| **NACHNAME** | Last name (may have typos/variations) |
| **Geburtsdatum** | Date of birth (YYYY-MM-DD) |
| **PLZ** | Postal code (may change over time = patient moved) |
| **GESCHLECHT** | Gender |
| **DWH_ZEITRAUM** | Quarter (YYYYQ format, e.g., 20231) |
| **Datenkoerper** | Source system |
| **ERGEBNISONLINEPRUEFUNG** | VSDM verification result |
| **ERRORCODE** | VSDM error code |
| **golden_*** | Ground truth "best" values for validation |

## Usage

- **Record Linkage**: Use `entity_id` to evaluate clustering accuracy
- **Survivorship**: Compare selected values against `golden_*` columns


In [1]:
import json
import random
import string
from datetime import date, timedelta
from typing import Optional

import numpy as np
import pandas as pd

# === CONFIGURATION ===
NUM_ENTITIES = 100_000       # Number of unique individuals
PARAMS_FILE = "synthesis_params.json"
OUTPUT_FILE = "patients_gen.csv"
RANDOM_SEED = 42

# Temporal settings
MIN_QUARTERS = 2            # Minimum quarters of history per entity
MAX_QUARTERS = 8            # Maximum quarters of history per entity
START_QUARTER = 20201       # First quarter (2020 Q1)
END_QUARTER = 20244         # Last quarter (2024 Q4)

# Change/error probabilities
ADDRESS_CHANGE_PROB = 0.15  # 15% chance patient moved between quarters
TYPO_PROB = 0.20            # 20% of records have typos (linkage challenge)
KVNR_MISSING_PROB = 0.20    # 20% records missing KVNR (overridden by params)
FIELD_MISSING_PROBS = {     # Probability each field is missing
    "VORNAME": 0.01,
    "NACHNAME": 0.01,
    "PLZ": 0.05,
    "GESCHLECHT": 0.03,
}

# VSDM verification (defaults - overridden by params if available)
VSDM_COVERAGE_RATE = 0.60   # 60% of records have any VSDM check
VSDM_VALID_GIVEN_CHECK = 0.85  # 85% of VSDM checks are valid (Tier 1)

# Source system simulation
SOURCE_SYSTEMS = ["BEARBEITET", "UNBEARBEITET", "KVUEPP"]
SOURCE_WEIGHTS = [0.6, 0.3, 0.1]

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Calculate expected output size
avg_quarters = (MIN_QUARTERS + MAX_QUARTERS) / 2
expected_rows = int(NUM_ENTITIES * avg_quarters)

print("=== UNIFIED DATASET CONFIGURATION ===")
print(f"Unique entities: {NUM_ENTITIES:,}")
print(f"Quarters per entity: {MIN_QUARTERS}-{MAX_QUARTERS}")
print(f"Time range: Q{START_QUARTER%10}/{START_QUARTER//10} - Q{END_QUARTER%10}/{END_QUARTER//10}")
print(f"Address change probability: {ADDRESS_CHANGE_PROB*100:.0f}%")
print(f"Typo probability: {TYPO_PROB*100:.0f}%")
print(f"VSDM coverage: ~{VSDM_COVERAGE_RATE*100:.0f}% (default, loaded from params)")
print(f"Expected total rows: ~{expected_rows:,}")


=== UNIFIED DATASET CONFIGURATION ===
Unique entities: 100,000
Quarters per entity: 2-8
Time range: Q1/2020 - Q4/2024
Address change probability: 15%
Typo probability: 20%
VSDM coverage: ~60% (default, loaded from params)
Expected total rows: ~500,000


## 1. Load Synthesis Parameters

Load the pre-computed statistics from `synthesis_params.json`.


In [2]:
with open(PARAMS_FILE, "r", encoding="utf-8") as f:
    params = json.load(f)

print(f"Loaded parameters from: {PARAMS_FILE}")
print(f"Source dataset size: ~{params['metadata']['source_rows_approx']:,} rows")
print(f"Privacy: k≥{params['metadata']['k_anonymity_threshold']}")


Loaded parameters from: synthesis_params.json
Source dataset size: ~23,637,100 rows
Privacy: k≥100


## 2. Distribution Samplers

Create weighted random samplers from the extracted distributions.


In [3]:
def create_weighted_sampler(distribution: list, value_key: str) -> tuple:
    """
    Create arrays for weighted random sampling from a distribution.
    
    Args:
        distribution: List of dicts with value and count keys
        value_key: Key name for the value field
        
    Returns:
        Tuple of (values array, probabilities array)
    """
    values = [d[value_key] for d in distribution]
    counts = [d["count"] for d in distribution]
    total = sum(counts)
    probs = [c / total for c in counts]
    return np.array(values), np.array(probs)


def sample_from_dist(values: np.ndarray, probs: np.ndarray, n: int) -> np.ndarray:
    """Sample n values using weighted probabilities."""
    return np.random.choice(values, size=n, p=probs)


In [4]:
# Load name distributions
vorname_vals, vorname_probs = create_weighted_sampler(
    params["valid_data_distributions"]["vorname"]["frequency_distribution"],
    "VORNAME"
)
nachname_vals, nachname_probs = create_weighted_sampler(
    params["valid_data_distributions"]["nachname"]["frequency_distribution"],
    "NACHNAME"
)

# Load KVNR first letter distribution
kvnr_letter_vals, kvnr_letter_probs = create_weighted_sampler(
    params["valid_data_distributions"]["kvnr"]["first_letter_distribution"],
    "first_letter"
)

# Load PLZ regional distribution (first 2 digits)
plz_dist = params["valid_data_distributions"]["plz"]["regional_distribution"]
valid_plz_dist = [d for d in plz_dist if d["plz_region"].isdigit()]
plz_region_vals, plz_region_probs = create_weighted_sampler(valid_plz_dist, "plz_region")

# Load birth year distribution (5-year bins)
birth_year_dist = params["valid_data_distributions"]["geburtsdatum"]["birth_year_bins"]
valid_birth_years = [d for d in birth_year_dist if 1900 <= d["birth_year_bin"] <= 2025]
birth_year_vals, birth_year_probs = create_weighted_sampler(valid_birth_years, "birth_year_bin")

# Load month and day distributions
month_vals, month_probs = create_weighted_sampler(
    params["valid_data_distributions"]["geburtsdatum"]["month_distribution"], "month"
)
day_vals, day_probs = create_weighted_sampler(
    params["valid_data_distributions"]["geburtsdatum"]["day_distribution"], "day"
)

# Load gender distribution (if available)
if "geschlecht" in params["valid_data_distributions"]:
    gender_dist = params["valid_data_distributions"]["geschlecht"]["distribution"]
    gender_vals, gender_probs = create_weighted_sampler(gender_dist, "GESCHLECHT")
    print(f"  • Gender values: {len(gender_vals)} values (from params)")
else:
    # Fallback: simple distribution
    gender_vals = np.array(["M", "W", "0"])
    gender_probs = np.array([0.48, 0.48, 0.04])
    print(f"  • Gender values: {len(gender_vals)} values (defaults)")

# Load VSDM distribution (if available)
if "vsdm" in params["valid_data_distributions"]:
    vsdm_params = params["valid_data_distributions"]["vsdm"]
    vsdm_result_dist = vsdm_params["result_distribution"]
    vsdm_result_vals, vsdm_result_probs = create_weighted_sampler(
        vsdm_result_dist, "ERGEBNISONLINEPRUEFUNG"
    )
    # Override defaults with actual statistics
    VSDM_VALID_RATE_FROM_PARAMS = vsdm_params.get("valid_rate", VSDM_VALID_GIVEN_CHECK)
    VSDM_COVERAGE_RATE_FROM_PARAMS = vsdm_params.get("coverage_rate", VSDM_COVERAGE_RATE)
    
    # Load TEMPORAL VSDM coverage (CRITICAL: older quarters have no VSDM)
    VSDM_TEMPORAL_COVERAGE = {}  # quarter -> coverage percentage
    if "temporal_coverage" in vsdm_params:
        for item in vsdm_params["temporal_coverage"]:
            q = int(item["DWH_ZEITRAUM"])
            cov = item["coverage_pct"] / 100.0  # Convert to fraction
            VSDM_TEMPORAL_COVERAGE[q] = cov
        print(f"  • VSDM temporal coverage loaded: {len(VSDM_TEMPORAL_COVERAGE)} quarters")
    
    VSDM_START_QUARTER = vsdm_params.get("start_quarter", 20201)  # Default to 2020 Q1
    
    print(f"  • VSDM result values: {len(vsdm_result_vals)} values")
    print(f"  • VSDM valid rate: {VSDM_VALID_RATE_FROM_PARAMS*100:.1f}% (among VSDM records)")
    print(f"  • VSDM overall coverage: {VSDM_COVERAGE_RATE_FROM_PARAMS*100:.1f}%")
    print(f"  • VSDM started from quarter: {VSDM_START_QUARTER}")
else:
    # Fallback: use defaults
    vsdm_result_vals = np.array([1, 2, 0, 3, 4, 5])
    vsdm_result_probs = np.array([0.45, 0.40, 0.05, 0.04, 0.03, 0.03])
    VSDM_VALID_RATE_FROM_PARAMS = VSDM_VALID_GIVEN_CHECK
    VSDM_COVERAGE_RATE_FROM_PARAMS = VSDM_COVERAGE_RATE
    VSDM_TEMPORAL_COVERAGE = {}  # Empty = use overall rate
    VSDM_START_QUARTER = 20201
    print(f"  • VSDM result values: {len(vsdm_result_vals)} values (defaults)")
    print(f"  • VSDM valid rate: {VSDM_VALID_RATE_FROM_PARAMS*100:.1f}% (default)")
    print(f"  • VSDM coverage rate: {VSDM_COVERAGE_RATE_FROM_PARAMS*100:.1f}% (default)")

# Also load KVNR error rate to adjust KVNR_MISSING_PROB
if "error_injection" in params and "field_errors" in params["error_injection"]:
    kvnr_error_rate = params["error_injection"]["field_errors"].get("KVNR", {}).get("error_rate", KVNR_MISSING_PROB)
    KVNR_MISSING_PROB_FROM_PARAMS = kvnr_error_rate
    print(f"  • KVNR missing rate: {KVNR_MISSING_PROB_FROM_PARAMS*100:.1f}% (from params)")
else:
    KVNR_MISSING_PROB_FROM_PARAMS = KVNR_MISSING_PROB
    print(f"  • KVNR missing rate: {KVNR_MISSING_PROB_FROM_PARAMS*100:.1f}% (default)")

print(f"\nLoaded distributions:")
print(f"  • First names: {len(vorname_vals):,} values")
print(f"  • Last names: {len(nachname_vals):,} values")
print(f"  • KVNR letters: {len(kvnr_letter_vals)} values")
print(f"  • PLZ regions: {len(plz_region_vals)} values")
print(f"  • Birth year bins: {len(birth_year_vals)} values")


  • Gender values: 5 values (from params)
  • VSDM result values: 6 values
  • VSDM valid rate: 87.6% (among VSDM records)
  • VSDM overall coverage: 58.0%
  • VSDM started from quarter: 20201
  • KVNR missing rate: 16.1% (from params)

Loaded distributions:
  • First names: 1,000 values
  • Last names: 1,000 values
  • KVNR letters: 26 values
  • PLZ regions: 100 values
  • Birth year bins: 26 values


## 3. KVNR Generator

Generate valid German health insurance numbers with proper checksums.


In [5]:
def calculate_kvnr_checksum(letter: str, serial: str) -> int:
    """
    Calculate KVNR check digit using official Modulo-10 algorithm.
    
    Args:
        letter: First letter (A-Z)
        serial: 8-digit serial number
        
    Returns:
        Check digit (0-9)
    """
    char_code = ord(letter.upper()) - 64
    validation_string = f"{char_code:02d}{serial}"
    weights = [1, 2] * 5
    
    total = 0
    for i, char in enumerate(validation_string):
        product = int(char) * weights[i]
        total += sum(int(d) for d in str(product))
    
    return total % 10


def generate_valid_kvnr(letter: str) -> str:
    """Generate a valid KVNR with correct checksum."""
    serial = "".join(str(random.randint(0, 9)) for _ in range(8))
    check_digit = calculate_kvnr_checksum(letter, serial)
    return f"{letter}{serial}{check_digit}"


def generate_invalid_kvnr_checksum(letter: str) -> str:
    """Generate a KVNR with intentionally wrong checksum."""
    serial = "".join(str(random.randint(0, 9)) for _ in range(8))
    correct = calculate_kvnr_checksum(letter, serial)
    wrong = (correct + random.randint(1, 9)) % 10
    return f"{letter}{serial}{wrong}"


# Test
print(f"Sample valid KVNR: {generate_valid_kvnr('A')}")
print(f"Sample invalid KVNR: {generate_invalid_kvnr_checksum('A')}")


Sample valid KVNR: A104332188
Sample invalid KVNR: A196001338


## 4. PLZ Generator

Generate German postal codes based on regional distribution.


In [6]:
def generate_plz(region: str) -> str:
    """Generate a 5-digit PLZ for a given 2-digit region."""
    suffix = "".join(str(random.randint(0, 9)) for _ in range(3))
    return f"{region}{suffix}"


def generate_invalid_plz() -> str:
    """Generate an invalid PLZ (wrong format or out of range)."""
    error_type = random.choice(["short", "out_of_range", "non_numeric"])
    if error_type == "short":
        return str(random.randint(1000, 9999))
    elif error_type == "out_of_range":
        return str(random.randint(0, 999)).zfill(5)
    else:
        return f"{random.randint(10, 99)}-{random.randint(100, 999)}"


# Test
print(f"Sample valid PLZ: {generate_plz('48')}")
print(f"Sample invalid PLZ: {generate_invalid_plz()}")


Sample valid PLZ: 48908
Sample invalid PLZ: 9928


## 5. Birth Date Generator

Generate dates based on year/month/day distributions.


In [7]:
def generate_birth_date(year_bin: int, month: int, day: int) -> str:
    """
    Generate a birth date string from distribution samples.
    
    Args:
        year_bin: 5-year bin start (e.g., 1980 means 1980-1984)
        month: Month (1-12)
        day: Day of month (1-31)
        
    Returns:
        Date string in YYYY-MM-DD format
    """
    year = year_bin + random.randint(0, 4)
    year = min(max(year, 1900), 2025)
    
    try:
        d = date(year, month, min(day, 28))
        try:
            d = date(year, month, day)
        except ValueError:
            pass
        return d.strftime("%Y-%m-%d")
    except ValueError:
        return f"{year}-{month:02d}-{min(day, 28):02d}"


def generate_invalid_date() -> str:
    """Generate an invalid or problematic date."""
    error_type = random.choice(["future", "ancient", "bad_format"])
    if error_type == "future":
        future = date.today() + timedelta(days=random.randint(30, 365))
        return future.strftime("%Y-%m-%d")
    elif error_type == "ancient":
        y = random.randint(1800, 1899)
        return f"{y}-{random.randint(1,12):02d}-{random.randint(1,28):02d}"
    else:
        y = random.randint(1950, 2000)
        return f"{y}/{random.randint(1,12)}/{random.randint(1,28)}"


# Test
print(f"Sample valid date: {generate_birth_date(1985, 6, 15)}")
print(f"Sample invalid date: {generate_invalid_date()}")


Sample valid date: 1988-06-15
Sample invalid date: 2026-08-17


## 6. Error Injection

Inject realistic errors based on observed error rates and types.


In [8]:
# Extract error rates from parameters
error_rates = {}
for field, stats in params["error_injection"]["field_errors"].items():
    error_rates[field] = stats["error_rate"]

print("Error rates by field:")
for field, rate in sorted(error_rates.items(), key=lambda x: x[1], reverse=True):
    print(f"  {field}: {rate*100:.2f}%")


Error rates by field:
  KVNR: 16.07%
  NACHNAME: 0.40%
  PLZ: 0.39%
  GEBURTSDATUM: 0.27%
  VORNAME: 0.14%


In [9]:
def inject_kvnr_error(letter: str) -> Optional[str]:
    """Generate a KVNR error. Returns None for missing, or invalid string."""
    if random.random() < 0.96:  # ~96% of KVNR errors are missing
        return None
    return generate_invalid_kvnr_checksum(letter)


def inject_name_error(valid_name: str) -> str:
    """Inject invalid characters into a name."""
    error_chars = [".", ",", "1", "2", "(", ")", "@", "#"]
    pos = random.randint(0, len(valid_name))
    char = random.choice(error_chars)
    return valid_name[:pos] + char + valid_name[pos:]


def inject_plz_error() -> Optional[str]:
    """Generate a PLZ error based on observed patterns."""
    r = random.random()
    if r < 0.04:  # ~4% missing
        return None
    elif r < 0.64:  # ~60% out of range
        return str(random.randint(0, 999)).zfill(5)
    else:
        return generate_invalid_plz()


def inject_date_error() -> Optional[str]:
    """Generate a date error based on observed patterns."""
    if random.random() < 0.02:  # ~2% missing
        return None
    y = random.randint(1800, 1899)
    return f"{y}-{random.randint(1,12):02d}-{random.randint(1,28):02d}"


## 7. Record Linkage Perturbations

Functions to create realistic variations of records for duplicate detection.


In [10]:
# Perturbation probabilities for each field when creating duplicates
PERTURB_PROBS = {
    "vorname": 0.3,      # 30% chance to modify first name
    "nachname": 0.3,     # 30% chance to modify last name
    "geburtsdatum": 0.1, # 10% chance to modify birth date
    "plz": 0.15,         # 15% chance to modify postal code
    "kvnr": 0.2,         # 20% chance to modify/drop KVNR
}

def perturb_name(name: str) -> str:
    """Apply realistic perturbation to a name."""
    if not name or len(name) < 2:
        return name
    
    perturbation = random.choice([
        "typo", "typo", "typo",  # Most common
        "abbreviate", "case", "missing", "swap_chars", "double_char"
    ])
    
    if perturbation == "typo":
        pos = random.randint(0, len(name) - 1)
        char = name[pos]
        if char.isalpha():
            nearby = {"a": "s", "e": "r", "i": "o", "o": "i", "u": "i",
                      "n": "m", "m": "n", "s": "a", "t": "r", "r": "t"}
            new_char = nearby.get(char.lower(), random.choice(string.ascii_lowercase))
            if char.isupper():
                new_char = new_char.upper()
            return name[:pos] + new_char + name[pos+1:]
    elif perturbation == "abbreviate":
        return name[0] + "."
    elif perturbation == "case":
        return name.upper() if name[0].islower() else name.lower()
    elif perturbation == "missing":
        return ""
    elif perturbation == "swap_chars" and len(name) > 2:
        pos = random.randint(0, len(name) - 2)
        return name[:pos] + name[pos+1] + name[pos] + name[pos+2:]
    elif perturbation == "double_char":
        pos = random.randint(0, len(name) - 1)
        return name[:pos] + name[pos] + name[pos:]
    
    return name


def perturb_date(date_str: str) -> str:
    """Apply realistic perturbation to a date."""
    if not date_str:
        return date_str
    
    perturbation = random.choice(["swap_day_month", "off_by_one", "typo", "missing"])
    
    try:
        parts = date_str.split("-")
        if len(parts) != 3:
            return date_str
        year, month, day = parts
        
        if perturbation == "swap_day_month":
            if int(day) <= 12:
                return f"{year}-{day}-{month}"
        elif perturbation == "off_by_one":
            field = random.choice(["day", "month", "year"])
            if field == "day":
                new_day = max(1, min(28, int(day) + random.choice([-1, 1])))
                return f"{year}-{month}-{new_day:02d}"
            elif field == "month":
                new_month = max(1, min(12, int(month) + random.choice([-1, 1])))
                return f"{year}-{new_month:02d}-{day}"
            else:
                new_year = int(year) + random.choice([-1, 1])
                return f"{new_year}-{month}-{day}"
        elif perturbation == "typo":
            pos = random.randint(0, len(date_str) - 1)
            if date_str[pos].isdigit():
                new_digit = str((int(date_str[pos]) + random.randint(1, 9)) % 10)
                return date_str[:pos] + new_digit + date_str[pos+1:]
        elif perturbation == "missing":
            return None
    except (ValueError, IndexError):
        pass
    
    return date_str


def perturb_plz(plz: str) -> str:
    """Apply realistic perturbation to a postal code."""
    if not plz:
        return plz
    
    perturbation = random.choice(["typo", "transpose", "missing", "truncate"])
    
    if perturbation == "typo":
        pos = random.randint(0, len(plz) - 1)
        if plz[pos].isdigit():
            new_digit = str((int(plz[pos]) + random.randint(1, 9)) % 10)
            return plz[:pos] + new_digit + plz[pos+1:]
    elif perturbation == "transpose" and len(plz) >= 2:
        pos = random.randint(0, len(plz) - 2)
        return plz[:pos] + plz[pos+1] + plz[pos] + plz[pos+2:]
    elif perturbation == "missing":
        return None
    elif perturbation == "truncate":
        return plz[:4]
    
    return plz


def perturb_kvnr(kvnr: str) -> Optional[str]:
    """Apply realistic perturbation to a KVNR."""
    if not kvnr:
        return kvnr
    
    perturbation = random.choice(["missing", "missing", "typo", "checksum_error"])
    
    if perturbation == "missing":
        return None
    elif perturbation == "typo" and len(kvnr) == 10:
        pos = random.randint(1, 9)
        if kvnr[pos].isdigit():
            new_digit = str((int(kvnr[pos]) + random.randint(1, 9)) % 10)
            return kvnr[:pos] + new_digit + kvnr[pos+1:]
    elif perturbation == "checksum_error" and len(kvnr) == 10:
        return generate_invalid_kvnr_checksum(kvnr[0])
    
    return kvnr


print("Perturbation functions loaded.")


Perturbation functions loaded.


## 8. Survivorship Generation Functions

Functions for generating temporal patient records with realistic value changes over time.
This supports survivorship validation by creating:
- Golden records (ground truth best values)
- Temporal variants with address changes, missing fields, VSDM variations


In [11]:
# === Quarter Utilities ===

def get_all_quarters(start: int, end: int) -> list:
    """Generate list of quarters between start and end (inclusive)."""
    quarters = []
    year = start // 10
    q = start % 10
    end_year = end // 10
    end_q = end % 10
    
    while year < end_year or (year == end_year and q <= end_q):
        quarters.append(year * 10 + q)
        q += 1
        if q > 4:
            q = 1
            year += 1
    return quarters


def quarter_diff(q1: int, q2: int) -> int:
    """Calculate number of quarters between two quarter codes."""
    y1, qtr1 = q1 // 10, q1 % 10
    y2, qtr2 = q2 // 10, q2 % 10
    return (y2 - y1) * 4 + (qtr2 - qtr1)


ALL_QUARTERS = get_all_quarters(START_QUARTER, END_QUARTER)
print(f"Available quarters: {len(ALL_QUARTERS)} ({ALL_QUARTERS[0]} to {ALL_QUARTERS[-1]})")


Available quarters: 20 (20201 to 20244)


In [12]:
# === Golden Record Generator ===

def generate_golden_record(entity_id: int) -> dict:
    """
    Generate the "golden" (ground truth) record for an entity.
    This represents the correct/best values that survivorship should select.
    """
    letter = np.random.choice(kvnr_letter_vals, p=kvnr_letter_probs)
    vorname = np.random.choice(vorname_vals, p=vorname_probs)
    nachname = np.random.choice(nachname_vals, p=nachname_probs)
    plz_region = np.random.choice(plz_region_vals, p=plz_region_probs)
    year_bin = int(np.random.choice(birth_year_vals, p=birth_year_probs))
    month = int(np.random.choice(month_vals, p=month_probs))
    day = int(np.random.choice(day_vals, p=day_probs))
    geschlecht = np.random.choice(gender_vals, p=gender_probs)
    
    return {
        "entity_id": entity_id,
        "golden_EGKVERSICHERTENNUMMER": generate_valid_kvnr(letter),
        "golden_VORNAME": vorname,
        "golden_NACHNAME": nachname,
        "golden_Geburtsdatum": generate_birth_date(year_bin, month, day),
        "golden_PLZ": generate_plz(plz_region),
        "golden_GESCHLECHT": geschlecht,
    }


def generate_address_history(golden_plz: str, n_quarters: int) -> list:
    """
    Generate address (PLZ) history with realistic changes.
    Returns list of PLZ values, one per quarter, with possible changes.
    """
    history = [golden_plz]
    current_plz = golden_plz
    
    for _ in range(n_quarters - 1):
        if random.random() < ADDRESS_CHANGE_PROB:
            # Patient moved - generate new PLZ
            new_region = np.random.choice(plz_region_vals, p=plz_region_probs)
            current_plz = generate_plz(new_region)
        history.append(current_plz)
    
    return history


print("Golden record generator loaded.")


Golden record generator loaded.


In [13]:
# === Temporal Variant Generator ===

def apply_name_variation(name: str) -> str:
    """
    Apply realistic name entry variation (not typos, but different entry styles).
    This simulates different data entry practices across sources/time.
    """
    if not name or random.random() > 0.2:  # 80% unchanged
        return name
    
    variation = random.choice(["case", "abbreviated", "extra_space"])
    
    if variation == "case":
        # Different capitalization: "MICHAEL" vs "Michael" vs "michael"
        return random.choice([name.upper(), name.lower(), name.title()])
    elif variation == "abbreviated" and len(name) > 3:
        # First name abbreviated: "Michael" -> "M." or "Mich."
        return random.choice([name[0] + ".", name[:4] + "."])
    elif variation == "extra_space":
        # Trailing/leading space (data entry artifact)
        return " " + name if random.random() < 0.5 else name + " "
    
    return name


def apply_field_missing(record: dict, quarter_idx: int, n_quarters: int) -> dict:
    """
    Apply missing field probabilities. Older records more likely to have missing data.
    Uses KVNR_MISSING_PROB_FROM_PARAMS loaded from synthesis_params.json.
    """
    # Older records (higher quarter_idx = more recent, so invert)
    age_factor = 1 + (n_quarters - quarter_idx - 1) * 0.1  # Older = higher factor
    
    for field, base_prob in FIELD_MISSING_PROBS.items():
        if field in record and random.random() < base_prob * age_factor:
            record[field] = None
    
    # KVNR missing rate from params (~16%) - no age_factor to match original distribution
    if random.random() < KVNR_MISSING_PROB_FROM_PARAMS:
        record["EGKVERSICHERTENNUMMER"] = None

    return record


def generate_vsdm_for_record(quarter: int, latest_quarter: int) -> tuple:
    """
    Generate VSDM verification data for a record.
    More recent records more likely to have valid VSDM.
    Uses rates loaded from synthesis_params.json.
    
    Returns: (ERGEBNISONLINEPRUEFUNG, ERRORCODE, ONLINEPRUEFUNGSDATUM)
    """
    # Check if this record has VSDM at all (use exact rate from params)
    if random.random() > VSDM_COVERAGE_RATE_FROM_PARAMS:
        return (None, None, None)
    
    # Has VSDM check - determine if valid (use rate from params)
    if random.random() < VSDM_VALID_RATE_FROM_PARAMS:
        # Valid VSDM (Tier 1 eligible)
        result = random.choice([1, 2])
        errorcode = None
        # Generate a plausible date within the quarter
        year = quarter // 10
        q = quarter % 10
        month = (q - 1) * 3 + random.randint(1, 3)
        day = random.randint(1, 28)
        check_date = f"{year}-{month:02d}-{day:02d}"
    else:
        # Invalid VSDM - sample from actual result distribution
        invalid_results = [v for v, p in zip(vsdm_result_vals, vsdm_result_probs) if v not in [1, 2]]
        result = int(np.random.choice(invalid_results)) if invalid_results else 0
        errorcode = random.choice([1, 2, 3, 100, 101, 200])
        check_date = None
    
    return (result, errorcode, check_date)


print("Temporal variant generator loaded.")


Temporal variant generator loaded.


In [ ]:
# === Unified Entity Generator ===

def generate_entity(entity_id: int) -> list:
    """
    Generate a complete patient entity with:
    - Golden record (ground truth for survivorship)
    - Multiple temporal variants across quarters
    - Realistic value changes (address moves)
    - Data entry errors/typos (for record linkage testing)
    - VSDM verification simulation
    - Missing field variations
    
    Returns list of records (dicts), each representing one observation.
    """
    # 1. Generate golden record (ground truth)
    golden = generate_golden_record(entity_id)
    
    # 2. Decide how many quarters of history
    n_quarters = random.randint(MIN_QUARTERS, MAX_QUARTERS)
    
    # 3. Select which quarters this entity appears in
    available_quarters = ALL_QUARTERS.copy()
    selected_quarters = sorted(random.sample(available_quarters, min(n_quarters, len(available_quarters))))
    latest_quarter = max(selected_quarters)
    
    # 4. Generate address history (PLZ changes over time)
    plz_history = generate_address_history(golden["golden_PLZ"], len(selected_quarters))
    
    # 5. Generate records for each quarter
    records = []
    for idx, quarter in enumerate(selected_quarters):
        # Base record from golden
        record = {
            "entity_id": entity_id,
            "DWH_ZEITRAUM": quarter,
            "Datenkoerper": np.random.choice(SOURCE_SYSTEMS, p=SOURCE_WEIGHTS),
            "EGKVERSICHERTENNUMMER": golden["golden_EGKVERSICHERTENNUMMER"],
            "VORNAME": apply_name_variation(golden["golden_VORNAME"]),
            "NACHNAME": apply_name_variation(golden["golden_NACHNAME"]),
            "Geburtsdatum": golden["golden_Geburtsdatum"],
            "PLZ": plz_history[idx],  # Address at this point in time
            "GESCHLECHT": golden["golden_GESCHLECHT"],
        }
        
        # Apply typos/data entry errors (for record linkage challenge)
        if random.random() < TYPO_PROB:
            record = apply_typo_perturbations(record)
        
        # Add VSDM verification
        vsdm_result, vsdm_error, vsdm_date = generate_vsdm_for_record(quarter, latest_quarter)
        record["ERGEBNISONLINEPRUEFUNG"] = vsdm_result
        record["ERRORCODE"] = vsdm_error
        record["ONLINEPRUEFUNGSDATUM"] = vsdm_date
        
        # Apply missing fields (more likely in older records)
        record = apply_field_missing(record, idx, len(selected_quarters))
        
        # Add golden record values for validation
        record.update(golden)
        
        records.append(record)
    
    return records


def apply_typo_perturbations(record: dict) -> dict:
    """
    Apply random typo/data entry errors to a record.
    This simulates real-world data entry mistakes for record linkage testing.
    """
    # Randomly pick which fields to perturb
    if random.random() < 0.5 and record.get("VORNAME"):
        record["VORNAME"] = perturb_name(record["VORNAME"])
    
    if random.random() < 0.5 and record.get("NACHNAME"):
        record["NACHNAME"] = perturb_name(record["NACHNAME"])
    
    if random.random() < 0.2 and record.get("Geburtsdatum"):
        record["Geburtsdatum"] = perturb_date(record["Geburtsdatum"])

    if random.random() < 0.3 and record.get("PLZ"):
        record["PLZ"] = perturb_plz(record["PLZ"])

    # Note: KVNR missing is handled by apply_field_missing with rate from params (~16%)
    # Removing KVNR perturbation here avoids compounding the missing rate

    return record


print("Unified entity generator loaded.")


Unified entity generator loaded.


## 8. Generate Synthetic Dataset

Combine all generators to create the full synthetic dataset.


In [15]:
def generate_base_entity(entity_id: int) -> dict:
    """Generate a base entity (unique person) with clean data."""
    letter = np.random.choice(kvnr_letter_vals, p=kvnr_letter_probs)
    vorname = np.random.choice(vorname_vals, p=vorname_probs)
    nachname = np.random.choice(nachname_vals, p=nachname_probs)
    plz_region = np.random.choice(plz_region_vals, p=plz_region_probs)
    year_bin = int(np.random.choice(birth_year_vals, p=birth_year_probs))
    month = int(np.random.choice(month_vals, p=month_probs))
    day = int(np.random.choice(day_vals, p=day_probs))
    geschlecht = np.random.choice(gender_vals, p=gender_probs)
    
    # VSDM: determine if this record has valid verification (use rate from params)
    has_valid_vsdm = random.random() < VSDM_VALID_RATE_FROM_PARAMS
    if has_valid_vsdm:
        vsdm_result = int(np.random.choice([1, 2]))
        vsdm_errorcode = None
    else:
        vsdm_result = int(np.random.choice(vsdm_result_vals, p=vsdm_result_probs))
        vsdm_errorcode = random.choice([0, 1, 2, 3, None])
    
    return {
        "entity_id": entity_id,
        "EGKVERSICHERTENNUMMER": generate_valid_kvnr(letter),
        "VORNAME": vorname,
        "NACHNAME": nachname,
        "Geburtsdatum": generate_birth_date(year_bin, month, day),
        "PLZ": generate_plz(plz_region),
        "GESCHLECHT": geschlecht,
        "ERGEBNISONLINEPRUEFUNG": vsdm_result,
        "ERRORCODE": vsdm_errorcode
    }


def create_duplicate(base: dict) -> dict:
    """Create a perturbed duplicate of a base entity."""
    dup = base.copy()
    
    # Apply perturbations based on probabilities
    if random.random() < PERTURB_PROBS["vorname"]:
        dup["VORNAME"] = perturb_name(dup["VORNAME"])
    
    if random.random() < PERTURB_PROBS["nachname"]:
        dup["NACHNAME"] = perturb_name(dup["NACHNAME"])
    
    if random.random() < PERTURB_PROBS["geburtsdatum"]:
        dup["Geburtsdatum"] = perturb_date(dup["Geburtsdatum"])
    
    if random.random() < PERTURB_PROBS["plz"]:
        dup["PLZ"] = perturb_plz(dup["PLZ"])
    
    if random.random() < PERTURB_PROBS["kvnr"]:
        dup["EGKVERSICHERTENNUMMER"] = perturb_kvnr(dup["EGKVERSICHERTENNUMMER"])
    
    return dup


In [16]:
def generate_dataset(n_entities: int) -> pd.DataFrame:
    """
    Generate a unified synthetic patient dataset for record linkage and survivorship.
    
    Each entity has:
    - Multiple temporal records across quarters
    - Typos/data entry errors (for linkage testing)
    - Address changes over time
    - VSDM verification data (for survivorship Tier 1)
    - Golden record ground truth
    
    Args:
        n_entities: Number of unique individuals
        
    Returns:
        DataFrame with entity_id for clustering and golden_* for survivorship
    """
    all_records = []
    total_records = 0
    entities_with_vsdm = 0
    entities_with_address_change = 0
    entities_with_typos = 0
    
    for entity_id in range(n_entities):
        # Generate all records for this entity
        entity_records = generate_entity(entity_id)
        all_records.extend(entity_records)
        total_records += len(entity_records)
        
        # Track statistics
        has_vsdm = any(r.get("ERGEBNISONLINEPRUEFUNG") in [1, 2] for r in entity_records)
        if has_vsdm:
            entities_with_vsdm += 1
        
        # Count address changes (different PLZ values)
        plz_values = [r.get("PLZ") for r in entity_records if r.get("PLZ")]
        if len(set(plz_values)) > 1:
            entities_with_address_change += 1
        
        # Check for typos (name differs from golden)
        golden_vorname = entity_records[0].get("golden_VORNAME")
        has_typo = any(r.get("VORNAME") != golden_vorname for r in entity_records if r.get("VORNAME"))
        if has_typo:
            entities_with_typos += 1
        
        # Progress update
        if (entity_id + 1) % 10000 == 0 or entity_id == n_entities - 1:
            pct = (entity_id + 1) / n_entities * 100
            print(f"Processed {entity_id + 1:,} / {n_entities:,} entities ({pct:.1f}%)")
    
    # Shuffle records so same-entity records aren't adjacent
    random.shuffle(all_records)
    
    print(f"\n✓ Entities: {n_entities:,}")
    print(f"✓ Total records: {total_records:,}")
    print(f"✓ Avg records per entity: {total_records/n_entities:.1f}")
    print(f"✓ Entities with valid VSDM (Tier 1): {entities_with_vsdm:,} ({entities_with_vsdm/n_entities*100:.1f}%)")
    print(f"✓ Entities with address changes: {entities_with_address_change:,} ({entities_with_address_change/n_entities*100:.1f}%)")
    print(f"✓ Entities with typos/variations: {entities_with_typos:,} ({entities_with_typos/n_entities*100:.1f}%)")
    
    return pd.DataFrame(all_records)


In [17]:
# Legacy functions removed - using unified generate_dataset() instead
print("Using unified generate_dataset() function")


Using unified generate_dataset() function


In [18]:
print("Generating unified dataset...")
print(f"  • {NUM_ENTITIES:,} unique entities")
print(f"  • {MIN_QUARTERS}-{MAX_QUARTERS} quarters per entity")
print(f"  • {ADDRESS_CHANGE_PROB*100:.0f}% address change probability")
print(f"  • {TYPO_PROB*100:.0f}% typo probability")
if VSDM_TEMPORAL_COVERAGE:
    min_cov = min(VSDM_TEMPORAL_COVERAGE.values()) * 100
    max_cov = max(VSDM_TEMPORAL_COVERAGE.values()) * 100
    print(f"  • VSDM coverage: {min_cov:.0f}%-{max_cov:.0f}% (varies by quarter)")
else:
    print(f"  • VSDM coverage: {VSDM_COVERAGE_RATE_FROM_PARAMS*100:.1f}%")
print(f"  • VSDM valid rate: {VSDM_VALID_RATE_FROM_PARAMS*100:.1f}%")
print("=" * 50)

synthetic_df = generate_dataset(NUM_ENTITIES)

print("=" * 50)
print(f"✓ Generation complete: {len(synthetic_df):,} total rows")


Generating unified dataset...
  • 100,000 unique entities
  • 2-8 quarters per entity
  • 15% address change probability
  • 20% typo probability
  • VSDM coverage: 58.0%
  • VSDM valid rate: 87.6%
Processed 10,000 / 100,000 entities (10.0%)
Processed 20,000 / 100,000 entities (20.0%)
Processed 30,000 / 100,000 entities (30.0%)
Processed 40,000 / 100,000 entities (40.0%)
Processed 50,000 / 100,000 entities (50.0%)
Processed 60,000 / 100,000 entities (60.0%)
Processed 70,000 / 100,000 entities (70.0%)
Processed 80,000 / 100,000 entities (80.0%)
Processed 90,000 / 100,000 entities (90.0%)
Processed 100,000 / 100,000 entities (100.0%)

✓ Entities: 100,000
✓ Total records: 500,505
✓ Avg records per entity: 5.0
✓ Entities with valid VSDM (Tier 1): 93,178 (93.2%)
✓ Entities with address changes: 52,403 (52.4%)
✓ Entities with typos/variations: 70,893 (70.9%)
✓ Generation complete: 500,505 total rows


## 9. Dataset Validation

Verify the unified dataset has features needed for both:
- **Record Linkage**: Multiple records per entity, typos/variations, cluster analysis
- **Survivorship**: Temporal distribution, VSDM coverage, golden record ground truth


In [19]:
# Dataset statistics
n_records = len(synthetic_df)
n_entities = synthetic_df["entity_id"].nunique()
cluster_sizes = synthetic_df["entity_id"].value_counts()

print("UNIFIED DATASET STATISTICS")
print("=" * 60)
print(f"Total records: {n_records:,}")
print(f"Unique entities: {n_entities:,}")
print(f"Avg records per entity: {n_records/n_entities:.2f}")

# Cluster size distribution (for record linkage)
print(f"\nCluster size distribution (record linkage):")
for size in sorted(cluster_sizes.unique())[:8]:
    count = (cluster_sizes == size).sum()
    print(f"  Size {size}: {count:,} entities ({count/n_entities*100:.1f}%)")

# Quarter distribution (for survivorship)
print(f"\nTemporal distribution (survivorship):")
quarter_counts = synthetic_df["DWH_ZEITRAUM"].value_counts().sort_index()
print(f"  Quarters covered: {quarter_counts.index.min()} - {quarter_counts.index.max()}")
print(f"  Records per quarter: {quarter_counts.mean():.0f} avg, {quarter_counts.min()}-{quarter_counts.max()} range")

# VSDM statistics
print(f"\nVSDM verification:")
vsdm_valid = synthetic_df["ERGEBNISONLINEPRUEFUNG"].isin([1, 2]).sum()
vsdm_any = synthetic_df["ERGEBNISONLINEPRUEFUNG"].notna().sum()
print(f"  Records with any VSDM: {vsdm_any:,} ({vsdm_any/n_records*100:.1f}%)")
print(f"  Records with valid VSDM (Tier 1): {vsdm_valid:,} ({vsdm_valid/n_records*100:.1f}%)")

# Source system distribution
print(f"\nSource system distribution:")
for source in SOURCE_SYSTEMS:
    count = (synthetic_df["Datenkoerper"] == source).sum()
    print(f"  {source}: {count:,} ({count/n_records*100:.1f}%)")


UNIFIED DATASET STATISTICS
Total records: 500,505
Unique entities: 100,000
Avg records per entity: 5.01

Cluster size distribution (record linkage):
  Size 2: 14,274 entities (14.3%)
  Size 3: 14,267 entities (14.3%)
  Size 4: 14,236 entities (14.2%)
  Size 5: 14,308 entities (14.3%)
  Size 6: 14,071 entities (14.1%)
  Size 7: 14,506 entities (14.5%)
  Size 8: 14,338 entities (14.3%)

Temporal distribution (survivorship):
  Quarters covered: 20201 - 20244
  Records per quarter: 25025 avg, 24768-25257 range

VSDM verification:
  Records with any VSDM: 290,460 (58.0%)
  Records with valid VSDM (Tier 1): 254,489 (50.8%)

Source system distribution:
  BEARBEITET: 300,320 (60.0%)
  UNBEARBEITET: 150,203 (30.0%)
  KVUEPP: 49,982 (10.0%)


In [20]:
# Show sample entities with temporal records and variations
print("SAMPLE ENTITY RECORDS")
print("=" * 80)

# Find entities with multiple records
entity_counts = synthetic_df["entity_id"].value_counts()
sample_entities = entity_counts[entity_counts >= 4].head(3).index.tolist()

for eid in sample_entities:
    entity_records = synthetic_df[synthetic_df["entity_id"] == eid].sort_values("DWH_ZEITRAUM")
    print(f"\n--- Entity {eid} ({len(entity_records)} records) ---")
    
    # Show golden values (ground truth for survivorship)
    golden_cols = [c for c in entity_records.columns if c.startswith("golden_")]
    print(f"Golden (ground truth): ", end="")
    golden_vals = entity_records[golden_cols].iloc[0].to_dict()
    print({k.replace("golden_", ""): v for k, v in golden_vals.items()})
    
    # Show temporal variants with variations
    display_cols = ["DWH_ZEITRAUM", "Datenkoerper", "VORNAME", "NACHNAME", "PLZ", 
                   "ERGEBNISONLINEPRUEFUNG", "ERRORCODE"]
    print("\nTemporal records (for record linkage clustering & survivorship selection):")
    print(entity_records[display_cols].to_string(index=False))
    
    # Highlight features
    plz_values = entity_records["PLZ"].dropna().unique()
    if len(plz_values) > 1:
        print(f"  📍 Address change detected: {list(plz_values)}")
    
    # Check for name variations/typos
    golden_vorname = entity_records["golden_VORNAME"].iloc[0]
    name_variations = entity_records["VORNAME"].dropna().unique()
    if len(name_variations) > 1 or (len(name_variations) == 1 and name_variations[0] != golden_vorname):
        print(f"  ✏️ Name variations: {list(name_variations)} (golden: {golden_vorname})")
    
    vsdm_valid = entity_records["ERGEBNISONLINEPRUEFUNG"].isin([1, 2]).sum()
    print(f"  📋 Valid VSDM records: {vsdm_valid}/{len(entity_records)}")


SAMPLE ENTITY RECORDS

--- Entity 154 (8 records) ---
Golden (ground truth): {'EGKVERSICHERTENNUMMER': 'G925010586', 'VORNAME': np.str_('Andrea'), 'NACHNAME': np.str_('SCHUMANN'), 'Geburtsdatum': '2023-06-04', 'PLZ': '44525', 'GESCHLECHT': np.str_('M')}

Temporal records (for record linkage clustering & survivorship selection):
 DWH_ZEITRAUM Datenkoerper VORNAME  NACHNAME   PLZ  ERGEBNISONLINEPRUEFUNG  ERRORCODE
        20201       KVUEPP  Andrea           44525                     2.0        NaN
        20203       KVUEPP  andrea        S. 44525                     NaN        NaN
        20212 UNBEARBEITET  Andrea  SCHUMANN 44525                     NaN        NaN
        20213   BEARBEITET  Andrea  SCHUMANN 44525                     1.0        NaN
        20223 UNBEARBEITET  Andrea  SCHUMANN 44525                     1.0        NaN
        20231   BEARBEITET Andrea   SCHUMANN 44525                     2.0        NaN
        20232   BEARBEITET  Andrea  SCHUMANN 44802                  

In [21]:
# Additional sample - show compact view of entity clusters
print("ENTITY CLUSTER SIZE SUMMARY")
print("=" * 60)
print(f"Min records per entity: {cluster_sizes.min()}")
print(f"Max records per entity: {cluster_sizes.max()}")
print(f"Median records per entity: {cluster_sizes.median():.1f}")


ENTITY CLUSTER SIZE SUMMARY
Min records per entity: 2
Max records per entity: 8
Median records per entity: 5.0


In [22]:
# Analyze data quality / perturbation effects
print("DATA QUALITY ANALYSIS")
print("=" * 60)

# Check for empty/missing values
for col in ["VORNAME", "NACHNAME", "Geburtsdatum", "PLZ", "EGKVERSICHERTENNUMMER", "GESCHLECHT"]:
    if col in synthetic_df.columns:
        null_count = synthetic_df[col].isna().sum()
        empty_count = (synthetic_df[col] == "").sum() if synthetic_df[col].dtype == object else 0
        total_missing = null_count + empty_count
        print(f"{col}: {total_missing:,} missing/empty ({total_missing/n_records*100:.2f}%)")

# VSDM analysis
print(f"\nVSDM analysis:")
print(f"VSDM checks: {synthetic_df['ERGEBNISONLINEPRUEFUNG'].notna().sum():,}")
print(f"VSDM errors: {(synthetic_df['ERRORCODE'].notna() & (synthetic_df['ERRORCODE'] != 0)).sum():,}")


DATA QUALITY ANALYSIS
VORNAME: 12,450 missing/empty (2.49%)
NACHNAME: 12,340 missing/empty (2.47%)
Geburtsdatum: 5,089 missing/empty (1.02%)
PLZ: 38,154 missing/empty (7.62%)
EGKVERSICHERTENNUMMER: 80,159 missing/empty (16.02%)
GESCHLECHT: 18,597 missing/empty (3.72%)

VSDM analysis:
VSDM checks: 290,460
VSDM errors: 35,971


In [23]:
# Compute statistics for both record linkage and survivorship evaluation
from math import comb

# === RECORD LINKAGE STATISTICS ===
print("PAIR STATISTICS (for record linkage evaluation)")
print("=" * 60)

# Number of true matching pairs (within same entity)
true_pairs = sum(comb(size, 2) for size in cluster_sizes if size > 1)

# Total possible pairs
total_pairs = comb(n_records, 2)
non_match_pairs = total_pairs - true_pairs

print(f"Total possible pairs: {total_pairs:,}")
print(f"True matching pairs: {true_pairs:,} ({true_pairs/total_pairs*100:.4f}%)")
print(f"Non-matching pairs: {non_match_pairs:,}")
print(f"Class imbalance ratio: 1:{non_match_pairs//max(true_pairs,1)}")

# === SURVIVORSHIP STATISTICS ===
print("\nGOLDEN RECORD VALIDATION (for survivorship evaluation)")
print("=" * 60)

# Verify golden record consistency
golden_cols = [c for c in synthetic_df.columns if c.startswith("golden_")]
print(f"Golden record columns: {len(golden_cols)}")

# Sample: verify that all records for an entity have same golden values
sample_entity = synthetic_df["entity_id"].iloc[0]
entity_records = synthetic_df[synthetic_df["entity_id"] == sample_entity]
golden_consistent = entity_records[golden_cols].nunique().max() == 1
print(f"Golden values consistent within entity: {'✓ Yes' if golden_consistent else '✗ No'}")

# Address change statistics
def count_address_changes(group):
    plz_vals = group["PLZ"].dropna().unique()
    return len(plz_vals) > 1

entities_with_changes = synthetic_df.groupby("entity_id").apply(count_address_changes).sum()
print(f"Entities with address changes: {entities_with_changes:,} ({entities_with_changes/n_entities*100:.1f}%)")


PAIR STATISTICS (for record linkage evaluation)
Total possible pairs: 125,252,377,260
True matching pairs: 1,202,726 (0.0010%)
Non-matching pairs: 125,251,174,534
Class imbalance ratio: 1:104139

GOLDEN RECORD VALIDATION (for survivorship evaluation)
Golden record columns: 6
Golden values consistent within entity: ✓ Yes
Entities with address changes: 52,403 (52.4%)


C:\Users\yanni\AppData\Local\Temp\ipykernel_9256\354566851.py:39: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  entities_with_changes = synthetic_df.groupby("entity_id").apply(count_address_changes).sum()


In [24]:
# === VALIDATION: Compare generated statistics to synthesis_params.json ===
print("PARAMETER VALIDATION")
print("=" * 70)
print("Comparing generated dataset to expected statistics from synthesis_params.json\n")

# Helper to format comparison
def compare_stat(name: str, expected: float, actual: float, tolerance: float = 0.03) -> str:
    diff = abs(expected - actual)
    status = "✓" if diff <= tolerance else "✗"
    return f"{status} {name}: expected {expected*100:.2f}%, actual {actual*100:.2f}% (diff: {diff*100:.2f}%)"

# 1. KVNR missing rate
expected_kvnr_missing = params["error_injection"]["field_errors"]["KVNR"]["error_rate"]
actual_kvnr_missing = synthetic_df["EGKVERSICHERTENNUMMER"].isna().sum() / len(synthetic_df)
print(compare_stat("KVNR missing rate", expected_kvnr_missing, actual_kvnr_missing))

# 2. VSDM coverage rate
expected_vsdm_coverage = params["valid_data_distributions"]["vsdm"]["coverage_rate"]
actual_vsdm_coverage = synthetic_df["ERGEBNISONLINEPRUEFUNG"].notna().sum() / len(synthetic_df)
print(compare_stat("VSDM coverage", expected_vsdm_coverage, actual_vsdm_coverage, tolerance=0.15))

# 3. VSDM valid rate (among records with VSDM)
expected_vsdm_valid = params["valid_data_distributions"]["vsdm"]["valid_rate"]
vsdm_records = synthetic_df[synthetic_df["ERGEBNISONLINEPRUEFUNG"].notna()]
actual_vsdm_valid = vsdm_records["ERGEBNISONLINEPRUEFUNG"].isin([1, 2]).sum() / len(vsdm_records)
print(compare_stat("VSDM valid rate (among VSDM records)", expected_vsdm_valid, actual_vsdm_valid))

# 4. Gender distribution (top 2)
gender_dist = params["valid_data_distributions"]["geschlecht"]["distribution"]
expected_gender = {d["GESCHLECHT"]: d["frequency_pct"] / 100 for d in gender_dist}
actual_gender = synthetic_df["GESCHLECHT"].value_counts(normalize=True).to_dict()
print("\nGender distribution:")
for g in ["W", "M"]:
    if g in expected_gender and g in actual_gender:
        print(f"  {compare_stat(f'Gender {g}', expected_gender[g], actual_gender[g], tolerance=0.05)}")

# 5. VORNAME error rate (as proxy for overall name quality)
# Note: We track missing/empty, which should be low
expected_vorname_error = params["error_injection"]["field_errors"]["VORNAME"]["error_rate"]
actual_vorname_missing = (synthetic_df["VORNAME"].isna() | (synthetic_df["VORNAME"] == "")).sum() / len(synthetic_df)
print(f"\nField quality:")
print(compare_stat("VORNAME missing/empty", expected_vorname_error, actual_vorname_missing, tolerance=0.03))

# 6. PLZ error rate
expected_plz_error = params["error_injection"]["field_errors"]["PLZ"]["error_rate"]
actual_plz_missing = synthetic_df["PLZ"].isna().sum() / len(synthetic_df)
# Note: We only track missing, not invalid format
print(f"  PLZ missing: {actual_plz_missing*100:.2f}% (params includes format errors too: {expected_plz_error*100:.2f}%)")

# 7. GEBURTSDATUM error rate  
expected_dob_error = params["error_injection"]["field_errors"]["GEBURTSDATUM"]["error_rate"]
actual_dob_missing = synthetic_df["Geburtsdatum"].isna().sum() / len(synthetic_df)
print(f"  Geburtsdatum missing: {actual_dob_missing*100:.2f}% (params includes invalid dates: {expected_dob_error*100:.2f}%)")

print("\n" + "-" * 70)
print("Note: Some differences are expected due to:")
print("  • Typo perturbations (for record linkage challenge)")
print("  • Age factor for older records")
print("  • Different validation criteria (missing vs invalid)")
print("-" * 70)


PARAMETER VALIDATION
Comparing generated dataset to expected statistics from synthesis_params.json

✓ KVNR missing rate: expected 16.07%, actual 16.02% (diff: 0.05%)
✓ VSDM coverage: expected 57.99%, actual 58.03% (diff: 0.04%)
✓ VSDM valid rate (among VSDM records): expected 87.63%, actual 87.62% (diff: 0.01%)

Gender distribution:
  ✓ Gender W: expected 58.78%, actual 58.63% (diff: 0.15%)
  ✓ Gender M: expected 41.17%, actual 41.33% (diff: 0.16%)

Field quality:
✓ VORNAME missing/empty: expected 0.14%, actual 2.49% (diff: 2.35%)
  PLZ missing: 7.62% (params includes format errors too: 0.39%)
  Geburtsdatum missing: 1.02% (params includes invalid dates: 0.27%)

----------------------------------------------------------------------
Note: Some differences are expected due to:
  • Typo perturbations (for record linkage challenge)
  • Age factor for older records
  • Different validation criteria (missing vs invalid)
----------------------------------------------------------------------


## 10. Export Dataset


In [25]:
import os

# Export to CSV
synthetic_df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")
print(f"✓ Exported to: {OUTPUT_FILE}")

# Calculate file size
file_size = os.path.getsize(OUTPUT_FILE) / (1024 * 1024)
print(f"  File size: {file_size:.2f} MB")


✓ Exported to: patients_gen.csv
  File size: 57.38 MB


In [26]:
# Final summary
print("\n" + "=" * 70)
print("UNIFIED PATIENT DATASET READY")
print("=" * 70)
print(f"\n✓ Total records: {len(synthetic_df):,}")
print(f"✓ Unique entities: {n_entities:,}")
print(f"✓ Avg records per entity: {len(synthetic_df)/n_entities:.1f}")
print(f"✓ True matching pairs: {true_pairs:,}")
print(f"✓ Output file: {OUTPUT_FILE}")

print(f"\nData columns: {[c for c in synthetic_df.columns if not c.startswith('golden_')]}")
print(f"Ground truth: {[c for c in synthetic_df.columns if c.startswith('golden_')]}")

print("\n" + "-" * 70)
print("DATASET FEATURES")
print("-" * 70)
print("  • entity_id: Ground truth clustering (same ID = same person)")
print("  • DWH_ZEITRAUM: Temporal quarter for each observation")
print("  • Datenkoerper: Source system (BEARBEITET/UNBEARBEITET/KVUEPP)")
print("  • Typos & variations: Names may differ from golden (linkage challenge)")
print("  • Address changes: PLZ varies over time for some entities")
print("  • VSDM fields: ERGEBNISONLINEPRUEFUNG, ERRORCODE (for Tier 1 selection)")
print("  • golden_* columns: Ground truth for survivorship validation")

print("\n" + "-" * 70)
print("USAGE")
print("-" * 70)
print("Record Linkage:")
print("  • Splink: Use entity_id as 'cluster' for training")
print("  • dedupe: Use entity_id to create labeled pairs")
print("  • Evaluation: Compare predicted clusters to entity_id")
print("\nSurvivorship:")
print("  • Test Tier 1 (VSDM) selection: Filter ERGEBNISONLINEPRUEFUNG in [1,2]")
print("  • Test Tier 2 (score-based) selection: Time-weighted frequency scoring")
print("  • Validate: Compare selected values to golden_* columns")

print("=" * 70)



UNIFIED PATIENT DATASET READY

✓ Total records: 500,505
✓ Unique entities: 100,000
✓ Avg records per entity: 5.0
✓ True matching pairs: 1,202,726
✓ Output file: patients_gen.csv

Data columns: ['entity_id', 'DWH_ZEITRAUM', 'Datenkoerper', 'EGKVERSICHERTENNUMMER', 'VORNAME', 'NACHNAME', 'Geburtsdatum', 'PLZ', 'GESCHLECHT', 'ERGEBNISONLINEPRUEFUNG', 'ERRORCODE', 'ONLINEPRUEFUNGSDATUM']
Ground truth: ['golden_EGKVERSICHERTENNUMMER', 'golden_VORNAME', 'golden_NACHNAME', 'golden_Geburtsdatum', 'golden_PLZ', 'golden_GESCHLECHT']

----------------------------------------------------------------------
DATASET FEATURES
----------------------------------------------------------------------
  • entity_id: Ground truth clustering (same ID = same person)
  • DWH_ZEITRAUM: Temporal quarter for each observation
  • Datenkoerper: Source system (BEARBEITET/UNBEARBEITET/KVUEPP)
  • Typos & variations: Names may differ from golden (linkage challenge)
  • Address changes: PLZ varies over time for some en